In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

In [ ]:
!pip install -q lightning
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U transformers peft
!pip install -q -U accelerate
!pip install -q -U huggingface_hub

In [ ]:
from huggingface_hub import login
login(token="Your-token-here")

In [ ]:
import gdown

csv_path_1 = "JL_final_df(14400).csv"
csv_path_2 = "SL_code_90k.csv"

if not os.path.exists(csv_path_1):
    gdown.download('https://drive.google.com/file/d/1sHDJBVBdut0O_oX6N7sjbLluAC2M4OPH/view?usp=sharing', csv_path_1, quiet=False, fuzzy=True)

if not os.path.exists(csv_path_2):
    gdown.download('https://drive.google.com/file/d/1x5rB7fp3WSQROnUT6SIOKhJxbbnCZb-u/view?usp=sharing', csv_path_2, quiet=False, fuzzy=True)

In [ ]:
import pandas as pd
import numpy as np

SL_df = pd.read_csv('SL_code_90k.csv')
JL_df = pd.read_csv('JL_final_df(14400).csv')
final_df = pd.concat([SL_df, JL_df], ignore_index=True)

score_min = final_df['score'].min()
score_max = final_df['score'].max()
if score_max > 1.0 or score_min < 0.0:
    print(f"Normalizing scores from [{score_min}, {score_max}] to [0, 1]")
    final_df['score'] = (final_df['score'] - score_min) / (score_max - score_min + 1e-9)

print(f"Score range after normalization: [{final_df['score'].min():.4f}, {final_df['score'].max():.4f}]")
print(f"Total samples: {len(final_df)}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, Trainer
from sklearn.model_selection import train_test_split
from torch import optim
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig, AutoConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training


class codedataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['content'])
        label = float(self.df.iloc[idx]['score'])
        label = max(0.0, min(1.0, label))

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return (
            encoding['input_ids'].flatten(),
            encoding['attention_mask'].flatten(),
            torch.tensor(label, dtype=torch.float)
        )


class codeModule(pl.LightningDataModule):
    def __init__(self, train_dataset, valid_dataset, batch_size=1, num_workers=2):
        super().__init__()
        self.train_dataset = train_dataset
        self.valid_dataset = valid_dataset
        self.batch_size = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.valid_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True)

In [ ]:
model_name = "codellama/CodeLlama-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

print(f"Tokenizer Pad Token: {tokenizer.pad_token}")
print(f"Tokenizer Pad ID: {tokenizer.pad_token_id}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

base_model = AutoModel.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
import time
import sys
from IPython.display import FileLink
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Callback
from huggingface_hub import HfApi, hf_hub_download


class CodeExperienceModel(pl.LightningModule):
    def __init__(self, model, lr=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['model'])
        self.model = model
        self.lr = lr

        self.model.gradient_checkpointing_enable()

        self.regressor = nn.Sequential(
            nn.Linear(4096, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

        self.loss_fn = nn.BCEWithLogitsLoss()

    def on_load_checkpoint(self, checkpoint):
        state_dict = checkpoint["state_dict"]
        quant_suffixes = (
            ".absmax", ".quant_map", ".nested_absmax",
            ".nested_quant_map", ".bitsandbytes__nf4"
        )
        checkpoint["state_dict"] = {
            k: v for k, v in state_dict.items()
            if not any(k.endswith(s) or "quant_state" in k for s in quant_suffixes)
        }

    def _pool(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask, 1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        return self.regressor(self._pool(input_ids, attention_mask))

    def training_step(self, batch, batch_idx):
        input_ids, attention_mask, labels = batch
        labels = labels.float().unsqueeze(1)
        embeddings = self._pool(input_ids, attention_mask)

        if np.random.random() > 0.5:
            lam = np.random.beta(1.0, 1.0)
            index = torch.randperm(embeddings.size(0)).to(self.device)
            mixed_embeddings = lam * embeddings + (1 - lam) * embeddings[index]
            mixed_labels = torch.clamp(lam * labels + (1 - lam) * labels[index], 0.0, 1.0)
            preds = self.regressor(mixed_embeddings)
            loss = self.loss_fn(preds, mixed_labels)
        else:
            preds = self.regressor(embeddings)
            loss = self.loss_fn(preds, labels)

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids, attention_mask, labels = batch
        labels = labels.float().unsqueeze(1)
        preds = self(input_ids, attention_mask)
        loss = self.loss_fn(preds, labels)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return optim.AdamW(self.parameters(), lr=self.lr)


class HuggingFaceTimerCallback(Callback):
    def __init__(self, repo_id, upload_interval_hours=2.0):
        super().__init__()
        self.repo_id = repo_id
        self.upload_interval_secs = upload_interval_hours * 3600
        self.last_upload_time = time.time()
        self.api = HfApi()
        self.api.create_repo(repo_id=self.repo_id, repo_type="model", exist_ok=True)

    def _print(self, msg):
        sys.__stdout__.write(msg + "\n")
        sys.__stdout__.flush()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        current_time = time.time()
        if current_time - self.last_upload_time >= self.upload_interval_secs:
            self._print(f"\n[Timer] {self.upload_interval_secs/3600:.2f} hours elapsed. Backing up to Hugging Face...")
            os.makedirs("checkpoints", exist_ok=True)
            ckpt_path = "checkpoints/last_timer_backup.ckpt"
            trainer.save_checkpoint(ckpt_path)
            try:
                self.api.upload_file(
                    path_or_fileobj=ckpt_path,
                    path_in_repo="last2.ckpt",
                    repo_id=self.repo_id,
                    repo_type="model"
                )
                self._print("Backup successful! Training will now resume.\n")
                self.last_upload_time = current_time
            except Exception as e:
                self._print(f"Backup failed (will retry next interval): {e}\n")


def save_and_push_model(model, repo_id, filename="Codellama_v2_0"):
    checkpoint_path = f"{filename}.pth"
    torch.save(model.state_dict(), checkpoint_path)
    zip_name = f"{filename}.zip"
    os.system(f"zip {zip_name} {checkpoint_path}")
    try:
        api = HfApi()
        api.upload_file(
            path_or_fileobj=checkpoint_path,
            path_in_repo=checkpoint_path,
            repo_id=repo_id,
            repo_type="model"
        )
        print("Successfully uploaded final weights to Hugging Face!")
    except Exception as e:
        print(f"Failed to upload to Hugging Face: {e}")
    return FileLink(zip_name)


full_dataset = codedataset(final_df, tokenizer)
train_ds, valid_ds = train_test_split(full_dataset, test_size=0.3, shuffle=True, random_state=42)
datamodule = codeModule(train_dataset=train_ds, valid_dataset=valid_ds, batch_size=1, num_workers=2)

lightning_model = CodeExperienceModel(peft_model)

early_stop_callback = EarlyStopping(monitor="val_loss", patience=3, mode="min")
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",
    filename="best-checkpoint",
    save_top_k=1,
    monitor="val_loss",
    mode="min",
    save_last=True,
    every_n_epochs=1
)

HF_REPO_ID = "sujalgawas/my-codellama-experience-model"
hf_timer_callback = HuggingFaceTimerCallback(repo_id=HF_REPO_ID, upload_interval_hours=2.0)

trainer = Trainer(
    callbacks=[early_stop_callback, checkpoint_callback, hf_timer_callback],
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=True,
    accumulate_grad_batches=8,
    enable_model_summary=False
)

try:
    print("Checking Hugging Face for an existing checkpoint...")
    downloaded_ckpt_path = hf_hub_download(repo_id=HF_REPO_ID, filename="last2.ckpt")
    print(f"Found checkpoint! Resuming from: {downloaded_ckpt_path}")
except Exception as e:
    print(f"No checkpoint found ({e}). Starting fresh!")
    downloaded_ckpt_path = None

trainer.fit(
    lightning_model,
    train_dataloaders=datamodule.train_dataloader(),
    val_dataloaders=datamodule.val_dataloader(),
    ckpt_path=downloaded_ckpt_path
)

save_and_push_model(lightning_model, repo_id=HF_REPO_ID)

In [ ]:
lightning_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lightning_model.to(device)

text = """public class AverageCalculator {
    def main(args):
        int sum = 0;
        for (i in range(0, len(args))) {
            sum += args[i];
        }
        console.log("Average = " + (sum / args.length));
        return sum / args.length;
}
"""

inputs = tokenizer(
    text,
    return_tensors='pt',
    truncation=True,
    max_length=512,
    padding='max_length'
)

with torch.no_grad():
    logits = lightning_model(
        inputs['input_ids'].to(device),
        inputs['attention_mask'].to(device)
    )
    score = torch.sigmoid(logits)
    print(f"Predicted Score (0=Intern, 1=Senior): {score.item():.4f}")